In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv("../output_data/survey/data_DESS-psychometrics_2025-09-17_09-49.csv", encoding="utf-16")

In [3]:
len(df)

159

In [ ]:
# only select particiapants with last page == 12
df = df.loc[df["LASTPAGE"] == 12].copy()

In [5]:
len(df)

150

## Attention Checks

In [6]:
ac1 = df.loc[df["SA01_23"] != 3, "WE08_RV1"].tolist()
ac2 = df.loc[df["SO01_23"] != 6, "WE08_RV1"].tolist()

# get number of prolific IDs, where people failed both attention checks ()
len(list(set(ac1) & set(ac2)))

0

In [7]:
# exlude participants from analysis that failed one out of the two attention checks
failed = ac1 + ac2
df = df.loc[~df["WE08_RV1"].isin(failed)].copy()

len(df)

144

In [8]:
# all based on US?
print("US-based:")
df.DE19.value_counts()

US-based:


DE19
 1.0    143
-9.0      1
Name: count, dtype: int64

In [9]:
# all have English as native language?
print("English as native language:")
df.DE22.value_counts()

English as native language:


DE22
1.0    144
Name: count, dtype: int64

## Demographics

In [10]:
# age
print("mean", df.DE02_01.mean())
print("std", df.DE02_01.std())

mean 47.763888888888886
std 15.707993649446905


In [11]:
# gender
df.DE01.value_counts().sort_index()

# 1: female
# 2: male
# 3: non-binary
# 4: other

DE01
1.0    78
2.0    63
4.0     3
Name: count, dtype: int64

In [12]:
# highest level of education
df.DE12.value_counts().sort_index()

# 1: less than high school
# 2: high school
# 3: some college
# 4: technifal certification
# 5: associate degreee
# 6: bachelor
# 7: machers
# 8: doctoral
# 9: professional degree


DE12
1.0     7
2.0    39
3.0    36
5.0    11
6.0    36
7.0    13
9.0     2
Name: count, dtype: int64

In [13]:
# employment status
df.DE14.value_counts().sort_index()

# 1: pupil
# 2: apprentice
# 3: student
# 4: full time
# 5: part time
# 6: self employed
# 7: seeking
# 8: homemaker
# 9: retired
# 10: other

DE14
-1.0      1
 3.0      2
 4.0     56
 5.0     16
 6.0     15
 7.0     16
 8.0     12
 9.0     23
 10.0     3
Name: count, dtype: int64

In [14]:
# ethnicity
df.DE20.value_counts().sort_index()

# 1: white
# 2: asian
# 3: native hawaiian
# 4: hispanic
# 5: AA
# 6: native american
# 7: two or more
# 8: other
# 9: unknown

DE20
-1.0      1
 1.0    109
 2.0      4
 4.0      5
 5.0     23
 7.0      2
Name: count, dtype: int64

## Consistency scores

In [15]:
# set up item pairs
sexism_pairs = [(f"SA01_{i:02d}", f"SO01_{i:02d}") for i in range(1, 23)]

relevance_pairs = [(f"MA01_{i:02d}", f"MO01_{i:02d}") for i in range(1, 16)]
judgement_pairs = [(f"MA02_{i:02d}", f"MO02_{i:02d}") for i in range(1, 16)]
morality_pairs = relevance_pairs + judgement_pairs

racism_pairs = [(f"RA{i:02d}", f"RO{i:02d}") for i in range(2, 10)]

In [16]:
# compare (with NaN==NaN also treated as match)
def compare(a, b):
    return (a == b) | (a.isna() & b.isna())

In [17]:
# sexism
for left, right in sexism_pairs:
    df[f"match_{left}_{right}"] = compare(df[left], df[right])

# Fraction of matches per row
df["sexism_matches"] = df[[f"match_{l}_{r}" for l, r in sexism_pairs]].mean(axis=1)

print("mean fraction sexism:", df["sexism_matches"].mean())
print("std fraction sexism:", df["sexism_matches"].std())

mean fraction sexism: 0.5432449494949494
std fraction sexism: 0.19030797661692164


In [18]:
# racism
for left, right in racism_pairs:
    df[f"match_{left}_{right}"] = compare(df[left], df[right])

# Fraction of matches per row
df["racism_matches"] = df[[f"match_{l}_{r}" for l, r in racism_pairs]].mean(axis=1)

print("mean fraction racism:", df["racism_matches"].mean())
print("std fraction racism:", df["racism_matches"].std())

mean fraction racism: 0.8333333333333334
std fraction racism: 0.17739371879672475


In [19]:
# morality
for left, right in morality_pairs:
    df[f"match_{left}_{right}"] = compare(df[left], df[right])

# Fraction of matches per row
df["morality_matches"] = df[[f"match_{l}_{r}" for l, r in morality_pairs]].mean(axis=1)

print("mean fraction morality:", df["morality_matches"].mean())
print("std fraction morality:", df["morality_matches"].std())

mean fraction morality: 0.5969907407407408
std fraction morality: 0.1781766056152419
